# Advanced OCR — Colab runner (v2, OXR-1.0)

This notebook is **only a runner**. All code lives in the GitHub repo
`greencube-dev-boop/advanced-ocr` (synced below with `git pull` — push updates to
GitHub and just rerun this notebook, never re-upload it). All data lives in your
Google Drive under **`OCR Engine/`**.

Pipeline v2: raw PDF → page-by-page → **text-only pages** keep the exact native
text layer → **visual pages** are parsed by **SoMarkAI/OXR-1.0** (tables → Markdown,
formulas → LaTeX, chemical equations → mhchem, chemical structures → SMILES,
code blocks → fences) → one Markdown per book in `OCR Engine/refined/`.

**No API keys are needed.** The default `oxr.mode=free_api` uses the free OXR API
(no sign-up; ≤10MB/≤20 pages per request — the pipeline chunks accordingly).
Self-hosting OXR (`oxr.mode=local`) needs a ≥24GB BF16 GPU — see the notes cell below.

**Rerunning this notebook resumes where it stopped** (state: `OCR Engine/config/books.json`).

In [ ]:
# 1) Dependencies (idempotent)
!pip install -q PyMuPDF requests tqdm
print("deps ok")

In [ ]:
# 2) Mount Google Drive and set the OCR Engine root
import os
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/OCR Engine"
os.makedirs(DRIVE_ROOT, exist_ok=True)   # creates raw/ config/ refined/ temp/ on first use
print("Drive root:", DRIVE_ROOT)
for sub in ("raw", "config", "refined", "temp"):
    os.makedirs(os.path.join(DRIVE_ROOT, sub), exist_ok=True)
print("raw/ is where you drop PDFs:", os.path.join(DRIVE_ROOT, "raw"))

In [ ]:
# 3) Sync the codebase from GitHub (clone once, then git pull on every run)
import os, subprocess, sys

REPO_URL = "https://github.com/greencube-dev-boop/advanced-ocr.git"
GITHUB_TOKEN = ""  # repo is private: paste a GitHub PAT with repo-read here (or make the repo public)
WORKDIR = "/content/advanced-ocr"

url = REPO_URL.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO_URL
if os.path.isdir(os.path.join(WORKDIR, ".git")):
    subprocess.run(["git", "-C", WORKDIR, "pull", "--ff-only", url], check=False)
else:
    subprocess.run(["git", "clone", url, WORKDIR], check=True)

os.environ["PYTHONPATH"] = os.path.join(WORKDIR, "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
sys.path.insert(0, os.path.join(WORKDIR, "src"))
import ocr_pipeline
print("pipeline", ocr_pipeline.__version__, "from", WORKDIR)

## About API keys and OXR modes

- **No keys are required.** The free OXR API (`oxr.mode=free_api`, default) needs no sign-up.
  Limits per request: 10 MB / 20 pages — the pipeline builds chunks that respect both
  (oversized single pages are sent as rendered PNGs). The free API is queued and best-effort.
- **Self-hosting OXR** (`oxr.mode=local` in `Drive/OCR Engine/config/pipeline.json`): run the
  [OXR services](https://github.com/SoMarkAI/OXR) yourself (Docker all-in-one or the three-service
  source setup) and point `oxr.local_base_url` at it (default `http://127.0.0.1:8000/v1`).
  Requirements: Linux, NVIDIA GPU **≥24 GB with BF16 support** (free Colab T4 = 16 GB — NOT
  sufficient; Colab Pro A100/L4-class runtimes work), CUDA 13.0 driver, Python 3.12.

In [ ]:
# 4) Sanity check: deps, drive, OXR reachability
!python -m ocr_pipeline doctor --drive-root "{DRIVE_ROOT}"
!python -m ocr_pipeline init-config --drive-root "{DRIVE_ROOT}"

In [ ]:
# 5) Run / resume the pipeline.
#    Drop PDFs into Drive -> "OCR Engine/raw/" first.
#    Finished pages are skipped automatically; interrupt anytime and rerun to resume.
!python -m ocr_pipeline run --drive-root "{DRIVE_ROOT}"

In [ ]:
# 6) Inspect resume state
!python -m ocr_pipeline status --drive-root "{DRIVE_ROOT}"

## Outputs

- Combined book Markdown: `Drive/OCR Engine/refined/<book>.md`
- Per-page trail (final / meta / chunk JSON+MD): `Drive/OCR Engine/temp/<book>/`
- Resume state: `Drive/OCR Engine/config/books.json` — settings: `Drive/OCR Engine/config/pipeline.json`

## Tips

- Settings (OXR mode/URL, chunking, classification thresholds, figure markers) are edited in
  `Drive/OCR Engine/config/pipeline.json` — no code changes needed.
- Reprocess a book from scratch (e.g. after an engine change): `!python -m ocr_pipeline reset mybook.pdf --drive-root "{DRIVE_ROOT}"` then run again.
- Process a single book: `!python -m ocr_pipeline run-book mybook.pdf --drive-root "{DRIVE_ROOT}"`.
